<img src="https://datasciencedegree.wisconsin.edu/wp-content/themes/data-gulp/images/logo.svg" width="300">

 # <center>DS 710 - Lesson 7</center>
<center>Web crawling in Python with BeautifulSoup </center>


This notebook will work through some of the basics of the BeautifulSoup package, including

1. Exploring websites to find data
2. Web scraping basics
3. Pulling specific elements
4. Plotting gathered data on a map

---

References for the material in this notebook include 
* posted readings and media as listed on Canvas

I have additional links throughout this notebook, too!

---

## 0. Import BeautifulSoup

Start by importing the BeautifulSoup and Requests packages. The Requests package allows us to request web pages from websites that we'll use BeautifulSoup to web scrape.

In [ ]:
from bs4 import BeautifulSoup as bs #import as 'bs' so don't have to type out BeautifulSoup everytime
import requests
import pandas as pd #pandas will be used later in the workbook

---

## 1. Exploring the website sourcecode

Before we begin pulling the data from the webiste, we first need to understand the structure of the site.  We'll use in-browser tools called "Developer tools" to see the structure of the site.

To access the developer tools in Chrome:

* Mac: Go to the menu by clicking "View" > "Developer" > "Developer Tools" or using the shortcut Cmd + Alt + I
* Windows/Linux: Click the menu button (three vertical dots) then selecting "More Tools" > "Developer Tools" or using the shortcut Ctrl + Shift + I
* If you use a different browser, a quick google search will tell you how to access the developer tools.


Our focus will be on the Elements tab. You can click on the arrows to expand and collapse the visible items.


To find a specific part of the web page on the Elements tab, you can right click on the item on the page, and it will become highlighted in the developer tool.

---

## 2. Download page source, store as variable

To get started, we'll save the URL that we want to scrape to a variable, request the URL to see if we can scrape it, save the request response, and check the request response.

A request response of 200 means the request for the URL successful.

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_volcanoes_in_Europe"
response = requests.get(url, headers = {'User-Agent': 'Mozilla/5.0'})
print(response)

Now we'll save the response content and parse it. The output should look similar to what you say when using the developer tools.

In [ ]:
soup = bs(response.content, features="lxml") #lxml is the parser
print(soup)

The soup output provides an organized layout of the HTML content that includes the tags of each item. We'll find out why that's important in the next section.

---

## 3. Pull the table from the page

When scraping a web page, you are most likely looking for a certain data piece. These data pieces are segmented using tags like `<p>` or `<td>` at the beginning and ending. The tags can contain attributes that give additional information like the class of an object.

Looking at the web page we checked earlier, we can tell it contains a table. With the help of the developer tools, we can see its class is "wikitable".

We'll parse the data into a BeautifulSoup object and use find() to extract the table tag.

In [ ]:
soup = bs(response.text, features="lxml") #parse data
table_data = soup.find('table', {'class':"wikitable"}) #extract table with wikitable class

Next we'll convert the HTML table into a dataframe and preview the first few rows.

[🔗 link to pandas documentation on read html](https://pandas.pydata.org/docs/reference/api/pandas.read_html.html)

In [ ]:
#read HTML table into a list of dataframe objects
from io import StringIO
list_of_data_frames = pd.read_html(StringIO(str(table_data)), encoding='utf-8')
list_of_data_frames

In [ ]:
df_raw = list_of_data_frames[0]

#look at the first 5 rows
df_raw.head(5)

---

## 4. Deal with a bad cell

Check out the `Coordinates` column, there's some interesting stuff there.  The first row has some extra stuff in it.  

In [ ]:
df_raw['Coordinates'][0]

There's still this extra whitespace character, `\ufeff`, in the strings.  Let's get rid of those:

In [ ]:
df_raw['Coordinates'] = df_raw['Coordinates'].str.replace(u'\ufeff','')

In [ ]:
# print and verify it's gone!
df_raw['Coordinates'][0]

Ok, now we have uniform data in our columns, and we can proceed in extracting coordinates to plot.

---

## 5. Extract latitude and longitude as numbers

As a reminder, we have a data frame `df_raw` that contains data as-downloaded, but it's steps away from plotting. 

Our `Coordinates` column is still too much.  It has both hours-minutes-degrees, and decimal coordinates.  We need just one, and we'll choose to use the decimal representation.

In [ ]:
 df_raw['Coordinates'].head(5)

But I also know that one of the volcanoes in this list, Cumbre Vieja, doesn't have the same format of coordinates (see [the webpage](https://en.wikipedia.org/wiki/List_of_volcanoes_in_Europe)).  

I'm choosing to throw away the lines of the table that don't already have the decimal representation, or are flat-out missing coordinates.  

In [ ]:
# step 1: throw away the nan's (missing coordinates)
df = df_raw[ ~pd.isna(df_raw['Coordinates'])]

# step 2: throw away rows where Coordinates doesn't have a slash in it
df = df[ df['Coordinates'].map(lambda s: '/' in s)].copy()
                        #                            ^^
                        #                         copying to prevent warnings about working with a slice

df.head(5)

Finally, I'm ready to extract the latitude and longitude from the `Coordinates` column

In [ ]:
temp = df['Coordinates'].map(lambda s: s.split('/')[1])

df[['Latitude', 'Longitude']] = temp.str.split(expand=True)
df['Latitude'] = pd.to_numeric( df['Latitude'].map(lambda s: s.replace('°N','') if 'N' in s else "-"+s.replace('°S','')) )
df['Longitude'] = pd.to_numeric( df['Longitude'].map(lambda s: s.replace('°E','') if 'E' in s else "-"+s.replace('°W','')) )

df.head()

I want to use the Elevation to set the size of the markers, but the column isn't numeric.  I'll make a new column.

In [ ]:
max_marker_size = 30
df['markersize'] = df['Elevation (m)'].apply(lambda x: int(x.split('m',1)[0].replace(',','')))

# do the scaling
df['markersize'] = df['markersize']/df['markersize'].max()*max_marker_size

# show the data
df.head()

---

## 6. Plotting the Data

What can we do with a data table of Europe's tallest volcanos? Plot them on a map!

Now that we have all the data, we'll turn it into a data frame.

Before we plot the dataframe you just built, we'll need to import a couple packages. 

ℹ️ I'm going to use `cartopy` to do the map plotting.  
* See [this documentation page about installing `cartopy`](https://scitools.org.uk/cartopy/docs/latest/installing.html#installing), and install cartopy on your machine.  If you need help installing, please post over at Piazza.
* See [this example](https://scitools.org.uk/cartopy/docs/latest/gallery/lines_and_polygons/features.html#sphx-glr-gallery-lines-and-polygons-features-py), which is super helpful for showing kinds of features.

(We used to use [GeoPandas](https://geopandas.org/en/stable/docs.html), but installation of that library can be surprisingly challenging, so we stopped using it for teaching).

In [ ]:
import matplotlib.pyplot as plt # for plotting, this is normal at this point

Now we can create the plot.

In [ ]:
#initalize an axis
import cartopy.crs as ccrs
import cartopy.feature as cfeature

#I'll start by making a `fig,ax` pair, with a `projection` from the `cartopy` library.
# See [this documentation]() for more on projections in cartopy.
ax = plt.axes(projection=ccrs.PlateCarree()) # tell pyplot what projection to use, enabling later `add_feature` options.


#Set the `extent` of the map.  
#These are limits on the coordinates to plot.  
#I decided on my limits by eyeballing from [this map](https://www.mapsofworld.com/lat_long/europe.html)
# unit is degrees.  first is x0,x1, then is y0,y1
ax.set_extent([-35,50,30,75], crs=ccrs.PlateCarree())  # you have to pass the `crs` argument to get the coordinates to match
ax.coastlines(alpha=0.5)

ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.25)

#plot points
df.plot(x="Longitude",y="Latitude",s="markersize",kind="scatter",title=f"Tallest volcanoes in Europe", ax=ax,color='r')


# Save the plot by calling plt.savefig() BEFORE plt.show()
plt.savefig('tallest_volcanos_in_europe_location.png',bbox_inches="tight",dpi=300)

plt.show() # rarely do this in .py scripts -- prefer to save to disk using `savefig`.

## 7. A bar plot of the heights of the volcanos

Let's wrap up by plotting a bar chart of the heights of the volcanos.  

Note that our heights are still just strings, so we need to convert them to numbers before we can actually make the plot.  Observe:

In [ ]:
df['Elevation (m)'].head(5)

My path:

1. Get the stuff before the `'m'`.
2. Convert to a number

I'll use a lambda to split and replace commas in one line, and pandas `to_numeric` to do the conversion.

In [ ]:
df['Elevation'] = pd.to_numeric(df['Elevation (m)'].map(lambda s: s.split('m')[0].strip().replace(',','')))
df['Elevation'].head(5)

In [ ]:
import seaborn as sns

df_sorted = df.sort_values(by='Elevation')
p = sns.barplot(data=df_sorted,y="Elevation",x="Name") # number of bins is the number
p.tick_params(axis='x', rotation=90)

plt.savefig('tallest_volcanos_in_europe_heights.png',bbox_inches="tight",dpi=300)

---

## When you're done

🧠 After completing the guided practice above, log onto Canvas to complete Quiz 7. You may wish to keep this notebook open while you work.

Credits:

* Grant Roth: Fall 2022
* Prof. silviana amethyst: Fall 2022, Spring 2023, Summer 2023, Fall 2023, Spring 2024
* Dr. Allison Beemer: Summer 2025